In [2]:

!pip install --upgrade quick-sentiments
# just in case you want to install the package from the local directory
#pip install .\dist\quick_sentiments-0.3.1-py3-none-any.whl


   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ----------------------------------- ---- 2.4/2.6 MB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 13.8 MB/s  0:00:00
  Attempting uninstall: quick-sentiments
    Found existing installation: quick-sentiments 0.3.2
    Uninstalling quick-sentiments-0.3.2:
      Successfully uninstalled quick-sentiments-0.3.2


In [2]:
import polars as pl

# here I have three python script I built to pre_process the data and running the pipeline
# you can find the code in the tools/preprocess.py file
# you can find  the code in the tools/pipeline.py file
# the pre_process function is used to clean the text data, there are various options available, please check the tools/preprocess.py file for details
# the run_pipeline function is used to run the sentimental analysis pipeline, it takes the training data and the vectorizer and machine learning methods as input, and returns the results
from quick_sentiments import pre_process
from quick_sentiments import run_pipeline
from quick_sentiments import make_predictions

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\meala\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\meala\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### Training Dataset


Name your data as Train.csv and place it in the Training Data folder. Or you can change the path in the code below.


In [3]:
# keep you training dataset in the training data folder
# this template uses csv files 
# column names can be set in Python but this template does not automatically update the column for the demo 
# however, the function will give you the option to tell column names for the text and label data

df_train = pl.read_csv("demo/training_data/train.csv",encoding='ISO-8859-1') 
print(f"Dataset shape: {df_train.shape[0]} rows and {df_train.shape[1]} columns")


Dataset shape: 162758 rows and 5 columns


### DEMO

In [4]:
df_train.head()
# randomly select only 10% of the data since the dataset is large
#RUN ONLY ONCE
df_train = df_train.sample(fraction=0.1, shuffle=True, seed=42) 
df_train

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment
str,str,bool,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE"""
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE"""
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE"""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE"""
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE"""
…,…,…,…,…
"""destiny_epic_the_joker_hanniba…","""Amber Frey""",true,"""Right up to its haunting final…","""POSITIVE"""
"""fortune_courageous""","""Katelyn Johnson""",true,"""Familiar and inessential, but …","""POSITIVE"""
"""cosmic_vito_corleone_captain_a…","""Taylor Abbott""",false,"""Little more than stock charact…","""NEGATIVE"""


The dataset is for training. The sentiments are already labeled. This will allow us to train a model that can predict sentiments on new data.


In [5]:
# you can use the pre_process function to clean the text data
response_column = "reviewText" # this is the column name for the text data, feel free to change it to your text column name
sentiment_column = "sentiment" # this is the column name for the sentiment data, feel free to change it to your sentiment column name
print(df_train[response_column][2:5])


shape: (3,)
Series: 'reviewText' [str]
[
	null
	"10 Cloverfield Lane is an exci…
	"It's funny, fast, and charming…
]


In [6]:
df_train[2:5]

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment
str,str,bool,str,str
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE"""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE"""
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE"""


In [7]:
pre_process(df_train[2:5], text_column=response_column, new_column_name="cleaned_text")

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment,cleaned_text
str,str,bool,str,str,str
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE""",""""""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE""","""cloverfield lane is an excitin…"
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE""","""it s funny fast and charming"""


In [ ]:
# make changes as necessary
# inside the map_elements, add  the parameters [pre_process(x, parameters_to_be_added)] and set it True/False if it differs from the defualt value
# check the tools/preprocess.py file for the parameters and their default values
# some of the parameters are remove_brackets, remove_stopwords, remove_punctuation, remove_numbers, remove_emojis, remove_urls, remove_html_tags, lemmatize, stem, lowercase
df_train = pre_process(df_train, text_column=response_column, new_column_name="cleaned_text")
df_train.head()

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment,cleaned_text
str,str,bool,str,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE""","""an acceptably mindless sanctua…"
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE""","""although many shots are out of…"
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE""",""""""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE""","""cloverfield lane is an excitin…"
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE""","""it s funny fast and charming"""


In [ ]:
#### in this template, there are four text representation / vectorizer methods available 
#### in the function run_pipeline (in python cell below), we shall make use of this, write the words inside [ ] for the methods you want to use
#### 1. Bag of Words [BOW] 
#### 2. Term Frequency [tf]
#### 3. TF -IDF    [tfidf]
#### 4. Word Embedding using Word2Vec (you can use other packages with slight changes) [wv] 
         # Word Embedding uses defualt 300 values; this will take some time to run

In [ ]:
#### in this template, there are also three machine learning methods that can be used
#### 1. Logistic Regression [logit]
#### 2. Random forest (recommended) (rf)
#### 3. XGBoosting  [XGB](word embedding and XGBoost may take long time to complete, combination of both is not recommended in local machine)

#I will keep this repository updated, and I will add more methods in the future

In [11]:
# this is the example of how to use the function
# you can change the vectorizer_name and model_name to the ones you want to use
# for now we will use word embedding and logistic regression
# write the name of your columns in the text_column_name and sentiment_column_name
# the text_column_name is the column name of the text data, and sentiment_column_name is

# run_pipeline function will return the dataframe with the vectorized text, vectorizer used  and the model
# it will also print the results of the model, including the accuracy and F1 score
# note, even without hyperparameter tuning, the model is getting over 70% accuracy in my test
# there may not be a need to perform hyperparameter tuning, but you can set perform_tuning to True if you want to do that

dt= run_pipeline(
    vectorizer_name="bow", # BOW, tf, tfidf, wv,  glove_25,glove_50, glove_100, gl0ve_200,
    model_name="logistic_regression", # logit, rf, XGB, nb, nn .#XGB takes long time, can not recommend using it on normal case
    df=df_train,
    text_column_name="cleaned_text",  # this is the column name of the text data, 
    sentiment_column_name = "sentiment",
    perform_tuning = False # make this true if you want to perform hyperparameter tuning, it will take longer time and 
                            # may run out of memory if the dataset is large,
)

# missing values in the text data will be removed

--- Running Pipeline for Bow + Logistic Regression ---
No missing values (None) found in text or sentiment columns. Proceeding with all rows.
Labels encoded: Original -> ['NEGATIVE' 'POSITIVE'], Encoded -> [0 1]
1. Splitting data into train/test...
2. Vectorizing  dataset (X)...
   - Generating Bag-of-Words features...
   - Transforming test data using fitted Bag-of-Words vectorizer...
3. Training and predicting...
   - Training Logistic Regression with default parameters (no hyperparameter tuning)...
   - Model trained with default parameters.
Best model parameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 100, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
4. Evaluating model...

Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.66      0.49      0.56      1072
    POSITIV

c:\Users\meala\anaconda3\envs\new_sentiments\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
## the dt is a dictionary that contains the results of the model, including the accuracy and F1 score
print(dt.keys())
# you can access the results using the keys of the dictionary
print("Vectorizer used: ", dt["vectorizer_name"])
print("Model used: ", dt["model_object"])
print("Accuracy: ", dt["accuracy"])



dict_keys(['model_object', 'vectorizer_name', 'vectorizer_object', 'label_encoder', 'y_test', 'y_pred', 'accuracy', 'report'])
Vectorizer used:  bow
Model used:  LogisticRegression(random_state=42)
Accuracy:  0.7471582181259601


### New Dataset for prediction
You can use the same format as the training dataset, but ensure that it contains the "Response" column for text data. The "Sentiment" column is optional for prediction datasets, as it will be generated by the model.
Make sure the dataset is saved in the "New Data" folder and is in CSV format.

In [13]:
new_data = pl.read_csv("demo/new_data/test.csv",encoding='ISO-8859-1') #keep your file here
print(new_data.shape)
new_data= new_data.sample(fraction=0.25, shuffle=True, seed=42)
print(new_data.shape)

(55315, 4)
(13828, 4)


In [14]:
new_data = pre_process(new_data, text_column=response_column, new_column_name="cleaned_text")
new_data.head()

movieid,reviewerName,isTopCritic,reviewText,cleaned_text
str,str,bool,str,str
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…"
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…"
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…"
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …"
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…"


In [16]:
make_predictions(
    new_data=new_data,
    text_column_name="cleaned_text",  # this is the column name of the text data,
    vectorizer=dt["vectorizer_object"],
    best_model=dt["model_object"],
    label_encoder=dt["label_encoder"],
    prediction_column_name="sentiment_predictions"  # Optional custom name
)

movieid,reviewerName,isTopCritic,reviewText,cleaned_text,sentiment_predictions
str,str,bool,str,str,str
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…","""POSITIVE"""
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…","""POSITIVE"""
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…","""NEGATIVE"""
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …","""POSITIVE"""
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…","""POSITIVE"""
…,…,…,…,…,…
"""katniss_everdeen_superman_harr…","""Bryan Phillips""",true,"""""No one's riding that loco thi…","""no one s riding that loco thin…","""NEGATIVE"""
"""evoke_wonder_woman_myriad_john…","""Michele Tucker""",true,"""[A Taste of Honey] has an eart…","""has an earthy gusto and sincer…","""POSITIVE"""
"""miracle_luke_skywalker_destiny…","""William Holland""",true,"""You put up with lines such as …","""you put up with lines such as …","""NEGATIVE"""
